# PyPSA network → Coupling input files

Combines a composed (not solved) PyPSA-Eur or PyPSA-Earth network with the generic input files of the coupling with LCA.

**Generic input files** (folder `generic_input_files/`, independent of any network):

| File                           | Content                                                                                    |
|--------------------------------|--------------------------------------------------------------------------------------------|
| `mapping.csv`                  | LCI datasets of the technologies, their components, resources and flows                    |
| `unit_conversion.xlsx`         | conversion factors with formulas and sources, **per unit of main output**                  |
| `technology_compositions.csv`  | sub-components of technologies if any (e.g., turbine and grid connection of wind turbines) |
| `technology_specifics.csv`     | technology-specific modifications of LCI datasets                                          |
| `lifetime.csv`                 | Lifetimes of infrastructure in LCI datasets                                                |
| `mapping_esm_flows_to_CPC.csv` | CPC categories correspnding to the PyPSA carriers                                          |

**Files built or completed from the network** (folder `your_specific_input_files/`): `model.csv` (flows of each technology), `efficiency.csv` (main input of combustion technologies), `lifetime.csv` (`ESM` column added, i.e., lifetimes of infrastructures in PyPSA), and the generic files restricted to the technologies of the network. All files are expressed **per unit of main output** (mescal convention), and `technologies.csv` gives the factors to convert the LCA indicators to the PyPSA capacity units.

**Scope.** The LCA scope is aligned with the CO2 scope of PyPSA: every component emitting CO2 to the atmosphere in PyPSA has an LCI dataset including these direct emissions, including the final demands burning fuels (agriculture, industry) and transport (aviation, shipping, land transport, with the manufacturing of vehicles), process emissions and the end of life of plastics. LCA indicators can thus be used alone, and their direct CO2 compared with the CO2 balance of PyPSA. Carbon capture is accounted where it happens: `gas for industry CC` is the industrial gas combustion with carbon capture (`Add CC` specific), next to `gas for industry` without capture, so that the avoided emissions appear in the LCA scores.

Technology categories:

| Category | Rule | In `model.csv` |
|---|---|---|
| `generation` | Generator with an investment cost | yes |
| `conversion` | Link with an investment cost, converting one carrier into another | yes |
| `resource` | Generator without investment cost and with a fuel cost (fuel supply) | yes |
| `end use` | Link without investment cost delivering a fuel to a final demand where it is burned, or releasing CO2 (process emissions, plastics), i.e., emitting CO2 in PyPSA (or burning a fuel listed in `end_use_fuels`), except those in `covered_elsewhere` | yes |
| `demand` | Final demand (Load) with an LCI dataset in the generic files, e.g., transport (see Final demands) | yes |
| `storage` | Store, StorageUnit, Link named `... charger` / `... discharger`, with an investment cost | no |
| `transmission` | Line, Link whose main input and output carriers are the same (pipelines) | no |
| `accounting` | Other components without investment cost (vents, ambient heat, EV chargers, storage chargers whose cost is on the store, ...) | no |

## Parameters

In [1]:
import ast
import logging
import os
import warnings
from pathlib import Path

import numpy as np
import openpyxl
import pandas as pd
import pypsa

logging.getLogger('pypsa').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', category=FutureWarning)

year = 2050
network_path = f'../resources/test-sector-perfect/networks/composed_{year}.nc'
lca_data_path = 'generic_input_files/'  # generic files
output_path = 'your_specific_input_files/'
default_lifetime = 25  # years, for technologies without lifetime in the network nor in the generic files
emission_flows = ['co2']  # CO2 atmosphere bus carrier, never chosen as main input or output
end_use_fuels = ['solid biomass']  # fuels delivered to final demands without CO2 flow in PyPSA (biogenic, carbon-neutral):
# their combustion is in the LCA scope, to balance the CO2 uptake of the biomass resource
covered_elsewhere = ['oil refining']  # emissions already in the LCI dataset of another technology (oil primary resource)
waste_heat_flow = 'waste heat'  # heat input of capture units assumed to use local waste heat in PyPSA-Eur
waste_heat_input = {'process emissions CC': 0.66}  # MWh per t CO2 captured (heat input of cement capture, technology-data)
max_efficiency = 100  # time-varying efficiencies equal to 0 or above this value mark unavailable snapshots
combustion_fuels = ['gas', 'oil', 'oil primary', 'coal', 'lignite', 'solid biomass', 'biogas', 'methanol',
                    'non-sequestered HVC']  # main inputs whose consumption drives direct emissions
Path(f'{output_path}/{year}').mkdir(parents=True, exist_ok=True)

## Functions

### Flows of each asset

One row per asset and port. A Link takes its input from `bus0` and delivers to `bus1`, `bus2`, ... with `efficiency`, `efficiency2`, ...; negative efficiencies are additional inputs. Three PyPSA-Eur conventions are handled:

- Links flagged `reversed` are copies of transmission links for the other direction: they are dropped.
- Links that can only run backwards (`p_max_pu <= 0`, e.g., heat pumps sized on their heat output) have their signs
  flipped, so that inputs are always negative and outputs positive.
- Time-varying efficiencies (heat pump COPs, thermal storage) are averaged over the snapshots where the technology
  is available: unavailable snapshots have an efficiency of 0 or a prohibitive value (e.g., 1000).

In [2]:
list_names = {
    'Generator': 'generators',
    'Link': 'links',
    'Store': 'stores',
    'StorageUnit': 'storage_units',
    'Line': 'lines',
}


def time_averaged_efficiency(
        static: pd.Series,
        dynamic: pd.DataFrame,
        weights: pd.Series,
) -> pd.Series:
    """
    Average the time-varying efficiencies over the snapshots where the asset is available

    :param static: static efficiency of each asset
    :param dynamic: time-varying efficiency of some assets (snapshots x assets)
    :param weights: snapshot weightings
    :return: efficiency of each asset (NaN if the asset is never available)
    """
    efficiency = static.astype(float).copy()
    if dynamic.empty:
        return efficiency
    available = dynamic.where((dynamic != 0) & (dynamic.abs() <= max_efficiency))
    available_weights = available.notna().mul(weights, axis=0).sum()
    efficiency.update(available.fillna(0).mul(weights, axis=0).sum() / available_weights.replace(0, np.nan))
    efficiency[available_weights.index[available_weights == 0]] = np.nan
    return efficiency


def get_asset_flows(n: pypsa.Network) -> pd.DataFrame:
    """
    Get the flows of each asset of the network, with inputs < 0 and outputs > 0 (per unit of bus0 for Links)

    :param n: PyPSA network
    :return: dataframe with one row per asset and port
    """
    weights = n.snapshot_weightings.generators
    bus_carrier = n.buses.carrier
    asset_flows = []

    for component in ['Generator', 'Store', 'StorageUnit', 'Line']:
        df = getattr(n, list_names[component])
        bus = df['bus0'] if component == 'Line' else df['bus']
        asset_flows.append(pd.DataFrame({
            'Component': component, 'Carrier': df.carrier, 'Asset': df.index, 'Port': 1,
            'Flow': bus.map(bus_carrier), 'Amount': 1.0, 'Static amount': 1.0,
        }))

    links = n.links[~n.links.reversed.fillna(False).astype(bool)] if 'reversed' in n.links else n.links
    sign = np.where(links.p_max_pu <= 0, -1.0, 1.0)  # links that only run backwards
    ports = sorted(int(col[3:]) for col in links.columns if col.startswith('bus') and col[3:].isdigit())
    for port in ports:
        if port == 0:
            amount = static_amount = pd.Series(-1.0, index=links.index)
        else:
            col = 'efficiency' if port == 1 else f'efficiency{port}'
            dynamic = n.links_t[col] if col in n.links_t else pd.DataFrame()
            amount = time_averaged_efficiency(links[col], dynamic[dynamic.columns.intersection(links.index)], weights)
            static_amount = links[col]
        asset_flows.append(pd.DataFrame({
            'Component': 'Link', 'Carrier': links.carrier, 'Asset': links.index, 'Port': port,
            'Flow': links[f'bus{port}'].map(bus_carrier), 'Amount': amount * sign, 'Static amount': static_amount * sign,
        }))

    asset_flows = pd.concat(asset_flows, ignore_index=True)
    asset_flows = asset_flows[asset_flows['Flow'].notna() & (asset_flows['Static amount'] != 0)]
    return asset_flows

### Technology names

A technology is a PyPSA carrier of a given component, so that LCA indicators can be assigned to the assets of the network from their component and carrier only. A name used by several components (e.g., the `gas` Store and the `gas` Generator) or identical to a flow (e.g., the `AC` lines and the `AC` electricity flow) gets the component in brackets: `gas (Store)`, `gas (Generator)`, `AC (Line)`.

If the assets of a carrier are connected to different carriers on the same port (e.g., DAC taking heat from urban central heat at some nodes and from urban decentral heat at others), the flow is renamed after the common part of the carrier names (`heat`), which has its own CPC categories in the generic files. If some assets have a port that others do not have (e.g., the waste heat of electrolysis, only used where there is district heating), the flow is averaged over the assets that have it.

Known limitation: assets of a carrier are assumed to be the same technology. This is not the case of `H2 Store`, which is a salt cavern (lifetime of 100 years in PyPSA-Eur) where caverns are available and a steel tank (30 years) elsewhere. All H2 stores are mapped to steel tanks, which overestimates the construction impacts of salt caverns (conservative assumption).

In [3]:
def add_technology_names(asset_flows: pd.DataFrame) -> pd.DataFrame:
    """
    Add the technology name of each asset, and harmonize the flows of assets of a same carrier

    :param asset_flows: flows of each asset
    :return: flows of each asset with a 'Name' column
    """
    asset_flows = asset_flows.copy()

    # Ports connected to different carriers: flow renamed after the common end of the carrier names (e.g., heat)
    for _, df in asset_flows.groupby(['Component', 'Carrier', 'Port']):
        flows = df['Flow'].unique()
        if len(flows) > 1:
            common = os.path.commonprefix([flow.split()[::-1] for flow in flows])[::-1]  # common last words
            asset_flows.loc[df.index, 'Flow'] = ' '.join(common) if common else df['Flow'].mode()[0]

    # Names shared by several components or identical to a flow (bus carrier) get the component in brackets
    asset_flows['Name'] = asset_flows['Carrier']
    components_per_name = asset_flows.groupby('Name')['Component'].nunique()
    shared_names = list(components_per_name.index[components_per_name > 1]) + list(asset_flows['Flow'].unique())
    shared = asset_flows['Name'].isin(shared_names)
    asset_flows.loc[shared, 'Name'] += ' (' + asset_flows.loc[shared, 'Component'] + ')'
    return asset_flows

### Technologies

Flows are averaged over the assets of each technology. The main output is the first positive port in the order
`bus1, bus0, bus2, ...` and the main input the first negative port in the order `bus0, bus1, ...`, skipping emission
flows.

End uses (e.g., `gas for industry`: gas → gas for industry, with the CO2 of its combustion) have the fuel delivered to
the final demand as main output: their LCI dataset is the combustion of that fuel (e.g., in an industrial furnace),
whose fuel input is removed by the double counting and whose direct emissions are kept. Links delivering fuels to
final demands with an LCI dataset (e.g., transport) are not end uses: their combustion is part of the final
demand technologies (see Final demands). Links releasing
CO2 only (`process emissions`, `HVC to air`) have the CO2 emitted as main output. In PyPSA-Eur, the CO2 of biogenic
fuels is not counted (carbon neutral), so `solid biomass for industry` has no CO2 flow; it is an end use anyway
(`end_use_fuels`), because its combustion emits the biogenic CO2 taken up by the biomass resource in the LCA. `oil
refining` stays out of the scope (`covered_elsewhere`), as refining is included in the LCI dataset of `oil primary`.

PyPSA measures the capacity and the dispatch of a Link on its `bus0` flow (`Reference flow`), e.g., MW and MWh of gas
for a CCGT, while all mescal inputs are per unit of main output. `Conversion` is the factor to apply to the LCA
indicators when integrating them in PyPSA: the efficiency of Links (e.g., 0.58 MWh of electricity per MWh of gas for
CCGT), the mean length in km for pipelines and lines, and the mean `max_hours` for StorageUnits whose generic factor is per MWh of storage (e.g., batteries in
electricity-only networks; see `align_storage_units`). StorageUnits with a natural inflow (hydro reservoirs) are
generation technologies. The lifetime is taken from the extendable assets (existing plants often have an infinite lifetime), else from
all assets. Assets left with PyPSA's default lifetime (infinite, i.e., not set, as for existing nuclear and hydro
plants in an overnight run), and technologies whose assets have different lifetimes (`H2 Store`), have no lifetime
here: it is completed later from the generic files.

In [4]:
def get_main_flow(flows: pd.DataFrame, output: bool) -> tuple[int, str] or tuple[None, None]:
    """
    Get the port and name of the main output or main input of a technology

    :param flows: average flows of the technology, indexed by port
    :param output: if True, return the main output, else the main input
    :return: port and flow name of the main output or input
    """
    order = [1, 0, 2, 3, 4] if output else [0, 1, 2, 3, 4]
    candidates = flows[(flows['Amount'] > 0) == output].reindex(order).dropna(subset=['Flow'])
    preferred = candidates[~candidates['Flow'].isin(emission_flows)]
    candidates = preferred if not preferred.empty else candidates
    if candidates.empty:
        return None, None
    return candidates.index[0], candidates['Flow'].iloc[0]


def get_category(component: str, carrier: str, assets: pd.DataFrame, main_input: str, main_output: str,
                 inflow_units: list, emits: bool, demand_buses: list) -> str:
    """
    Get the category of a technology

    :param component: PyPSA component
    :param carrier: PyPSA carrier
    :param assets: static data of the assets of the technology
    :param main_input: main input flow
    :param main_output: main output flow
    :param inflow_units: StorageUnits with a natural inflow (e.g., hydro reservoirs)
    :param emits: True if the technology emits CO2 to the atmosphere in PyPSA
    :param demand_buses: carriers of the buses of final demands with an LCI dataset (e.g., land transport oil)
    :return: category of the technology
    """
    no_investment = (assets['capital_cost'] == 0).all()
    if component == 'Line':
        return 'transmission'
    elif component == 'StorageUnit' and assets.index.isin(inflow_units).any():
        return 'generation'  # hydro reservoirs: the inflow is converted into electricity
    elif component in ['Store', 'StorageUnit']:
        return 'accounting' if no_investment else 'storage'
    elif component == 'Generator':
        if (assets['p_max_pu'] <= 0).all():
            return 'accounting'  # e.g., heat vents
        elif no_investment and (assets['marginal_cost'] == 0).all():
            return 'accounting'  # free ambient heat, e.g., river water heat
        return 'resource' if no_investment else 'generation'
    elif no_investment:
        if main_output in demand_buses:
            return 'accounting'  # fuel delivered to a final demand with an LCI dataset (e.g., transport), which
            # includes its combustion
        if carrier not in covered_elsewhere and (emits or main_input in end_use_fuels):
            return 'end use'  # fuel burned in a final demand (aviation, shipping, industry, ...), or process emissions
        return 'accounting'
    elif carrier.endswith('charger'):
        return 'storage'
    elif main_input == main_output:
        return 'transmission'
    else:
        return 'conversion'


def get_technologies(n: pypsa.Network, asset_flows: pd.DataFrame, lca_names: list) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Get the list of technologies and their flows per unit of main output

    :param n: PyPSA network
    :param asset_flows: flows of each asset, with technology names
    :param lca_names: names available in the generic files
    :return: technologies dataframe and flows dataframe (Name, Flow, Amount)
    """
    bus_unit = n.buses.groupby('carrier')['unit'].first()
    is_mass = lambda flow: str(bus_unit.get(flow, '')).startswith('t')  # e.g., t_co2
    inflow = n.storage_units_t.inflow
    inflow_units = list(inflow.columns[inflow.sum() > 0]) if not inflow.empty else []
    demand_buses = list(n.buses.carrier[n.loads.bus[n.loads.carrier.isin(get_lca_loads(n, lca_names))]].unique())

    # Average flows over the assets of each technology (assets never available are skipped)
    flows = asset_flows.groupby(['Name', 'Component', 'Carrier', 'Port', 'Flow'], as_index=False).agg(
        Amount=('Amount', 'mean'), Static=('Static amount', 'mean'))
    flows['Amount'] = flows['Amount'].fillna(flows['Static'])

    technologies, model = [], []
    for (name, component, carrier), df in flows.groupby(['Name', 'Component', 'Carrier']):
        df = df.set_index('Port')
        assets = getattr(n, list_names[component]).loc[asset_flows.loc[asset_flows['Name'] == name, 'Asset'].unique()]
        output_port, main_output = get_main_flow(df, output=True)
        _, main_input = get_main_flow(df, output=False)
        emits = (df['Flow'].isin(emission_flows) & (df['Amount'] > 0)).any()
        category = get_category(component, carrier, assets, main_input, main_output, inflow_units, emits,
                                demand_buses)

        nom = {'Store': 'e_nom', 'Line': 's_nom'}.get(component, 'p_nom')
        extendable = assets[assets[f'{nom}_extendable']]
        lifetime = (extendable if not extendable.empty else assets)['lifetime']
        lifetime = lifetime[np.isfinite(lifetime)]
        if lifetime.nunique() > 1:  # different lifetimes between assets (e.g., H2 Store): taken from the LCA side
            lifetime = lifetime.iloc[:0]

        # Reference flow: capacities and dispatch of Links are measured on bus0, other components on their output
        reference_flow = df.loc[0, 'Flow'] if component == 'Link' else main_output
        reference_unit = 'ton' if is_mass(reference_flow) else 'megawatt hour'
        if component == 'Line':
            capacity_unit = 'megavolt ampere'
        elif component == 'Store':
            capacity_unit = reference_unit
        else:
            capacity_unit = 'ton per hour' if is_mass(reference_flow) else 'megawatt'

        # Units of the generic construction factors, and conversion factor to the network units
        conversion, conversion_source = 1.0, None
        if category == 'transmission':
            lca_capacity_unit = ('megavolt ampere kilometer' if component == 'Line'
                                 else 'ton per hour kilometer' if is_mass(main_output) else 'megawatt kilometer')
            conversion = assets['length'].mean()
            conversion_source = f'mean length of the assets ({conversion:.4g} km): PyPSA network'
        elif component == 'StorageUnit':
            lca_capacity_unit = 'megawatt'  # see align_storage_units for factors per megawatt hour
        elif component == 'Store':
            lca_capacity_unit = capacity_unit
        else:
            lca_capacity_unit = 'ton per hour' if is_mass(main_output) else 'megawatt'
            if component == 'Link' and reference_flow != main_output:
                conversion = abs(df.loc[output_port, 'Amount'])
                output_quantity = 't' if is_mass(main_output) else 'MWh'
                reference_quantity = 't' if is_mass(reference_flow) else 'MWh'
                conversion_source = (f'efficiency ({conversion:.4g} {output_quantity} {main_output} per '
                                     f'{reference_quantity} {reference_flow}): PyPSA network')

        technologies.append({
            'Name': name,
            'Component': component,
            'Category': category,
            'Carrier': carrier,
            'Main input': main_input,
            'Main output': main_output,
            'Output unit': 'ton' if is_mass(main_output) else 'megawatt hour',
            'Reference flow': reference_flow,
            'Reference unit': reference_unit,
            'Capacity unit': capacity_unit,
            'LCA capacity unit': lca_capacity_unit,
            'Conversion': conversion,
            'Conversion source': conversion_source,
            'Max hours': assets['max_hours'].mean() if component == 'StorageUnit' else np.nan,
            'Lifetime': lifetime.mean() if not lifetime.empty else np.nan,
            'Number of assets': len(assets),
        })

        if category in ['generation', 'conversion', 'resource', 'end use']:
            df = df.assign(Amount=df['Amount'] / df.loc[output_port, 'Amount'])  # main output = 1 (mescal convention)
            df = df.groupby('Flow', as_index=False)['Amount'].sum()  # ports with the same flow are summed
            if name in waste_heat_input:  # capture heat supplied by local waste heat (not modelled in PyPSA-Eur)
                df = pd.concat([df, pd.DataFrame({'Flow': [waste_heat_flow], 'Amount': [-waste_heat_input[name]]})])
            model.append(df[df['Amount'].abs() > 1e-9].assign(Name=name)[['Name', 'Flow', 'Amount']])

    technologies = pd.DataFrame(technologies)
    technologies.loc[technologies['Category'].isin(['resource', 'accounting', 'end use']), 'Lifetime'] = np.nan

    return technologies, pd.concat(model, ignore_index=True)

### Final demands

Final demands are Loads. They are LCA technologies only if the generic files have an LCI dataset for them, under
the name `carrier (Load)` (the carrier name alone refers to the Link of the same name, e.g., the end use
`gas for industry`): this is the case of transport demands (`land transport EV`, `land transport oil`, `shipping
methanol`, `kerosene for aviation`, ...), whose LCI datasets are complete transport datasets (vehicle manufacturing,
maintenance and end of life, infrastructure, energy, exhaust and non-exhaust emissions), with the same scope for all
powertrains, so that no pathway is favoured by its modelling scope. Other demands (electricity, heat, industry, ...)
are `accounting`, as their energy supply is already covered by the technologies of the network.

A final demand is named `carrier (Load)`, its input is the energy of its bus (e.g., `EV battery`), and its main output
is the final energy delivered, as a `carrier service` flow in MWh (e.g., `land transport EV service`). The conversion
into transport service (vehicle km, tkm, pkm) is in the generic unit conversion file, with the efficiencies of
PyPSA-Eur (e.g., 100 x 53.19 km per MWh for EVs, 100 x 16.07 km per MWh for combustion engines), so that all
powertrains of a mode deliver the same service, as in PyPSA-Eur. The input of a final demand is counted as a combustion
fuel (efficiency correction) if the Links delivering it emit CO2. These Links are then `accounting`, as the combustion
is in the LCI dataset of the final demand.

Final demands are fixed in PyPSA-Eur (including powertrain shares): they add constant terms to the indicators of the
system, which can't bias the optimisation as long as they are exogenous. `Annual activity` is the yearly final energy
of each demand.


In [5]:
def get_lca_loads(n: pypsa.Network, lca_names: list) -> list:
    """
    Get the carriers of the Loads that have an LCI dataset in the generic files

    :param n: PyPSA network
    :param lca_names: names available in the generic files
    :return: list of Load carriers
    """
    return [carrier for carrier in n.loads['carrier'].unique() if f'{carrier} (Load)' in lca_names]


def get_demands(n: pypsa.Network, lca_names: list) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Get the final demands (Loads) with an LCI dataset and their flows per MWh of final energy

    :param n: PyPSA network
    :param lca_names: names available in the generic files
    :return: technologies dataframe and flows dataframe (Name, Flow, Amount)
    """
    weights = n.snapshot_weightings.generators
    p_set = n.loads_t.p_set.reindex(columns=n.loads.index).fillna(n.loads['p_set'])
    annual_energy = p_set.mul(weights, axis=0).sum() * 8760 / weights.sum()  # MWh per year
    bus_carrier = n.buses['carrier']

    technologies, model = [], []
    for carrier in get_lca_loads(n, lca_names):
        loads = n.loads[n.loads['carrier'] == carrier]
        input_flow = bus_carrier[loads['bus'].iloc[0]]
        service = f'{carrier} service'

        # Combustion: the Links delivering the energy to the demand emit CO2 to the atmosphere
        combustion = False
        for port in range(1, 5):
            if f'bus{port}' not in n.links:
                continue
            feeding = n.links[n.links[f'bus{port}'].isin(loads['bus'])]
            for emission_port in range(1, 5):
                col = 'efficiency' if emission_port == 1 else f'efficiency{emission_port}'
                if f'bus{emission_port}' in feeding and col in feeding:
                    emitting = feeding[f'bus{emission_port}'].map(bus_carrier).isin(emission_flows) & (feeding[col] > 0)
                    combustion = combustion or emitting.any()

        technologies.append({
            'Name': f'{carrier} (Load)',
            'Component': 'Load',
            'Category': 'demand',
            'Carrier': carrier,
            'Main input': input_flow,
            'Main output': service,
            'Output unit': 'megawatt hour',
            'Reference flow': input_flow,
            'Reference unit': 'megawatt hour',
            'Conversion': 1.0,
            'Lifetime': np.nan,
            'Number of assets': len(loads),
            'Combustion': combustion,
            'Annual activity': annual_energy[loads.index].sum(),
        })
        model += [{'Name': f'{carrier} (Load)', 'Flow': service, 'Amount': 1.0},
                  {'Name': f'{carrier} (Load)', 'Flow': input_flow, 'Amount': -1.0}]

    return pd.DataFrame(technologies), pd.DataFrame(model, columns=['Name', 'Flow', 'Amount'])


### Generic files

A network technology is looked up in the generic files under its name, then under its PyPSA carrier, then as
`carrier (component)` (e.g., `gas (Store)` when the network has no gas Generator).

Construction factors of StorageUnits are per MW of power by default (e.g., hydro, PHS); when the generic factor is
per MWh of storage (e.g., batteries in electricity-only networks), `Conversion` becomes the mean `max_hours`.

In [6]:
def get_lca_names(technologies: pd.DataFrame, lca_names: list) -> pd.Series:
    """
    Get the name of each network technology in the generic files

    :param technologies: technologies of the network
    :param lca_names: names available in the generic files
    :return: series with the network names as index and the generic names as values (NaN if missing)
    """
    names = {}
    for _, tech in technologies.iterrows():
        candidates = [tech['Name'], tech['Carrier'], f"{tech['Carrier']} ({tech['Component']})"]
        names[tech['Name']] = next((name for name in candidates if name in lca_names), np.nan)
    return pd.Series(names)


def align_storage_units(technologies: pd.DataFrame, unit_conversion: pd.DataFrame, lca_names: pd.Series) -> pd.DataFrame:
    """
    Use max_hours as conversion factor for the StorageUnits whose generic construction factor is per MWh

    :param technologies: technologies of the network
    :param unit_conversion: generic unit conversion file
    :param lca_names: output of get_lca_names
    :return: updated technologies
    """
    technologies = technologies.copy()
    per_mwh = unit_conversion.loc[(unit_conversion['Type'] == 'Construction')
                                  & (unit_conversion['ESM'] == 'megawatt hour'), 'Name']
    for i, tech in technologies[technologies['Component'] == 'StorageUnit'].iterrows():
        if lca_names.get(tech['Name']) in per_mwh.values:
            technologies.loc[i, ['LCA capacity unit', 'Conversion', 'Conversion source']] = [
                'megawatt hour', tech['Max hours'], f"mean energy to power ratio ({tech['Max hours']:.4g} h): PyPSA network"]
    return technologies


def complete_lifetimes(technologies: pd.DataFrame, lifetime: pd.DataFrame, lca_names: pd.Series) -> pd.DataFrame:
    """
    Complete the lifetimes missing in the network (PyPSA's default lifetime is infinite, i.e., not set) with the LCA
    lifetime of the generic file, else with default_lifetime

    :param technologies: technologies of the network
    :param lifetime: generic lifetime file
    :param lca_names: output of get_lca_names
    :return: technologies with 'Lifetime' and 'Lifetime source' columns completed
    """
    technologies = technologies.copy()
    lca_lifetime = lifetime.set_index('Name')['LCA']
    with_lifetime = ~technologies['Category'].isin(['resource', 'accounting', 'end use', 'demand'])
    technologies['Lifetime source'] = np.where(technologies['Lifetime'].notna() & with_lifetime, 'network', None)
    for i, tech in technologies[technologies['Lifetime'].isna() & with_lifetime].iterrows():
        value = lca_lifetime.get(lca_names.get(tech['Name']), np.nan)
        if pd.notna(value):
            technologies.loc[i, ['Lifetime', 'Lifetime source']] = [value, 'LCA']
        else:
            technologies.loc[i, ['Lifetime', 'Lifetime source']] = [default_lifetime, 'default']
    for source in ['LCA', 'default']:
        names = technologies.loc[technologies['Lifetime source'] == source, 'Name'].tolist()
        if names:
            print(f'Lifetime missing or different between assets in the network, {source} lifetime used for: {names}')
    return technologies


def adapt_to_network(df: pd.DataFrame, lca_names: pd.Series, keep: list = None) -> pd.DataFrame:
    """
    Keep the rows of a generic file that concern the network, renamed with the network technology names (rows
    shared by several network technologies are duplicated)

    :param df: generic dataframe with a 'Name' column
    :param lca_names: output of get_lca_names
    :param keep: other names to keep without renaming (components, flows)
    :return: dataframe for the network
    """
    rows = [df[df['Name'] == lca_name].assign(Name=name) for name, lca_name in lca_names.dropna().items()]
    if keep is not None:
        rows.append(df[df['Name'].isin(keep)])
    return pd.concat(rows, ignore_index=True)

### Consistency checks

Warnings are raised for the technologies of the network (except `accounting`) missing from the generic files,
for mapped LCI datasets without unit conversion factor or lifetime, for LCI datasets left blank in the mapping file,
for unit mismatches between the generic files and the network, for input flows without CPC categories, and for
produced flows without `Flow` LCI dataset.

In the mapping file, `Resource` datasets are needed for the carriers that can be imported or extracted (Generators
without investment cost, e.g., `gas (Generator)`), and `Flow` datasets for the carriers that can be produced by the
technologies of the network (e.g., `AC`, `H2`, `gas`), to create new markets in the LCI database after the ESM run.
`Flow` datasets are also given for the fuels of combustion technologies (e.g., `solid biomass`), whose unit
conversion factors the efficiency correction needs; they are simply not used if the fuel is never produced.

In [7]:
def check_ready_to_use_files(
        technologies: pd.DataFrame,
        model: pd.DataFrame,
        lca_names: pd.Series,
        mapping: pd.DataFrame,
        unit_conversion: pd.DataFrame,
        lifetime: pd.DataFrame,
        compositions: pd.DataFrame,
        flows_to_cpc: pd.DataFrame,
) -> None:
    """
    Check that the technologies and flows of the network are covered by the generic files

    :param technologies: technologies of the network
    :param model: flows of the technologies of the network
    :param lca_names: output of get_lca_names
    :param mapping: generic mapping file
    :param unit_conversion: generic unit conversion file
    :param lifetime: generic lifetime file
    :param compositions: generic technology compositions file
    :param flows_to_cpc: generic mapping between flows and CPC categories
    :return: None (warnings are raised)
    """
    technologies = technologies[technologies['Category'] != 'accounting'].set_index('Name')

    missing = [name for name in technologies.index if pd.isna(lca_names[name])]
    if missing:
        warnings.warn(f'Technologies of the network missing from the generic files: {missing}')

    conversions = set(zip(unit_conversion['Name'], unit_conversion['Type']))
    for name, lca_name in lca_names.dropna().items():
        tech = technologies.loc[name]
        types = mapping.loc[mapping['Name'] == lca_name, 'Type'].tolist()
        if lca_name in compositions['Name'].values:
            types.append('Construction')
            for component in compositions.loc[compositions['Name'] == lca_name, 'Components'].iloc[0]:
                if component not in mapping['Name'].values:
                    warnings.warn(f'No LCI dataset in the mapping file for the component {component} of {name}')
                if (component, 'Construction') not in conversions:
                    warnings.warn(f'No unit conversion factor for the component {component} of {name}')
                if component not in lifetime['Name'].values:
                    warnings.warn(f'No LCA lifetime for the component {component} of {name}')
        for type_ in set(types):
            if (lca_name, type_) not in conversions:
                warnings.warn(f'No unit conversion factor for {name} ({type_})')
                continue
            esm_unit = unit_conversion.loc[(unit_conversion['Name'] == lca_name) & (unit_conversion['Type'] == type_),
                                           'ESM'].iloc[0]
            expected = tech['LCA capacity unit'] if type_ == 'Construction' else tech['Output unit']
            if esm_unit != expected:
                warnings.warn(f'ESM unit of {name} ({type_}) is {esm_unit} in the unit conversion file, but '
                              f'{expected} in the network')
        if 'Construction' in types and lca_name not in lifetime['Name'].values:
            warnings.warn(f'No LCA lifetime for {name}')

    incomplete = sorted(set(name for name, lca_name in lca_names.dropna().items()
                            if mapping.loc[mapping['Name'] == lca_name, 'Activity'].isna().any()))
    if incomplete:
        warnings.warn(f'LCI datasets to be completed in the mapping file for: {incomplete}')

    input_flows = model.loc[model['Amount'] < 0, 'Flow'].unique()
    missing_flows = [flow for flow in input_flows if flow not in flows_to_cpc['Flow'].values]
    if missing_flows:
        warnings.warn(f'Input flows missing from the ESM flows to CPC mapping file: {missing_flows}')

    producers = technologies.index[technologies['Category'].isin(['generation', 'conversion'])]
    produced_flows = model.loc[(model['Amount'] > 0) & model['Name'].isin(producers)
                               & ~model['Flow'].isin(emission_flows), 'Flow'].unique()
    flows_without_dataset = [flow for flow in produced_flows
                             if flow not in mapping.loc[mapping['Type'] == 'Flow', 'Name'].values]
    if flows_without_dataset:
        warnings.warn(f'Flows produced in the network without Flow LCI dataset (no new market will be created for '
                      f'them after the ESM run): {flows_without_dataset}')

## Run

In [8]:
mapping = pd.read_csv(f'{lca_data_path}mapping.csv')
unit_conversion = pd.read_excel(f'{lca_data_path}unit_conversion.xlsx')
lifetime = pd.read_csv(f'{lca_data_path}lifetime.csv')
compositions = pd.read_csv(f'{lca_data_path}technology_compositions.csv')
compositions['Components'] = compositions['Components'].apply(ast.literal_eval)
tech_specifics = pd.read_csv(f'{lca_data_path}technology_specifics.csv')
flows_to_cpc = pd.read_csv(f'{lca_data_path}mapping_esm_flows_to_CPC.csv')
available_names = (mapping.loc[~mapping['Type'].isin(['Flow', 'Other']), 'Name'].unique().tolist()
                   + compositions['Name'].tolist())

n = pypsa.Network(network_path)
asset_flows = add_technology_names(get_asset_flows(n))
technologies, model = get_technologies(n, asset_flows, available_names)
demand_technologies, demand_model = get_demands(n, available_names)
technologies = pd.concat([technologies, demand_technologies], ignore_index=True)
model = pd.concat([model, demand_model], ignore_index=True)

technologies['Category'].value_counts()

Category
conversion      36
accounting      28
storage         17
generation      10
resource         9
end use          8
demand           5
transmission     3
Name: count, dtype: int64

In [9]:
lca_technologies = technologies[technologies['Category'] != 'accounting']
lca_names = get_lca_names(lca_technologies, available_names)

technologies = align_storage_units(technologies, unit_conversion, lca_names)
technologies = complete_lifetimes(technologies, lifetime, lca_names)

check_ready_to_use_files(technologies, model, lca_names, mapping, unit_conversion, lifetime, compositions, flows_to_cpc)

Lifetime missing or different between assets in the network, LCA lifetime used for: ['AC (Line)', 'co2 stored (Store)', 'gas (Store)', 'gas pipeline', 'methanol (Store)', 'oil (Store)']
Lifetime missing or different between assets in the network, default lifetime used for: ['battery', 'coal (Store)', 'home battery (Store)', 'lignite (Store)', 'uranium (Store)']


C:\Users\matth\AppData\Local\Temp\ipykernel_58188\3065575646.py:28: UserWarning: Technologies of the network missing from the generic files: ['coal (Store)', 'home battery charger', 'lignite (Store)', 'uranium (Store)', 'urban central aquifer thermal energy storage charger', 'urban central aquifer thermal energy storage discharger']
  warnings.warn(f'Technologies of the network missing from the generic files: {missing}')
C:\Users\matth\AppData\Local\Temp\ipykernel_58188\3065575646.py:64: UserWarning: Input flows missing from the ESM flows to CPC mapping file: ['solid biomass import']
  warnings.warn(f'Input flows missing from the ESM flows to CPC mapping file: {missing_flows}')
C:\Users\matth\AppData\Local\Temp\ipykernel_58188\3065575646.py:72: UserWarning: Flows produced in the network without Flow LCI dataset (no new market will be created for them after the ESM run): ['gas for industry', 'solid biomass for industry']
  warnings.warn(f'Flows produced in the network without Flow LCI d

## mescal input files

- `model.csv`: flows of the generation, conversion, resource, end-use and final demand technologies.
- `efficiency.csv`: only for conversion, end-use and final demand technologies whose main input is a fuel listed in
  `combustion_fuels` (or, for final demands, delivered by Links emitting CO2). The
  fuel input of the LCI dataset is set to 0 by the double-counting removal anyway; the efficiency correction scales
  its direct emissions with the efficiency ratio between the ESM and the LCI dataset. This is meaningful for combustion
  and reforming (CO2, NOx, particulates scale with the fuel input), but not, e.g., for heat pumps (refrigerant leaks do
  not scale with the electricity input). If the LCI dataset has no fuel input (e.g., burden-free waste incineration),
  there is simply nothing to correct. For end uses with carbon capture (e.g., `gas for industry CC`), it also
  accounts for the extra fuel burned for the capture heat.
- `lifetime.csv`: the `ESM` column is added from the network, for technologies (not for their components). When the
  network has no lifetime (PyPSA's default, infinite, e.g., for existing nuclear and hydro plants in an overnight run),
  or when its assets have different lifetimes (`H2 Store`), the LCA lifetime of the generic file is used, so that
  mescal does not rescale the construction impacts; if it is missing too, `default_lifetime` is used. `Lifetime source` in `technologies.csv` records which one was used.
- `mapping.csv`, `unit_conversion.xlsx`, `technology_compositions.csv`, `technology_specifics.csv`,
  `mapping_esm_flows_to_CPC.csv`: generic files restricted to the network, with the network technology names.
  All unit conversion factors stay per unit of main output (e.g., per MW and MWh of electricity for a CCGT),
  consistently with `model.csv` and with mescal's conventions (efficiency and capacity factor harmonisations). The
  `Conversion` column of `technologies.csv` gives, for each technology, the factor to apply when integrating the LCA
  indicators in PyPSA: main output per unit of `bus0` for Links (efficiency), km per asset for pipelines and lines,
  `max_hours` for StorageUnits, 1 otherwise.

In [10]:
components = [component for name in lca_names.dropna() if name in compositions['Name'].values
              for component in compositions.loc[compositions['Name'] == name, 'Components'].iloc[0]]
model_flows = model['Flow'].unique().tolist()
tech_data = technologies.set_index('Name')

# Mapping, compositions, specifics and CPC categories
mapping_network = pd.concat([
    adapt_to_network(mapping[~mapping['Type'].isin(['Flow', 'Other'])], lca_names, keep=components),
    mapping[(mapping['Type'] == 'Flow') & mapping['Name'].isin(model_flows)],
], ignore_index=True)
compositions_network = adapt_to_network(compositions, lca_names)
tech_specifics_network = adapt_to_network(tech_specifics, lca_names)
flows_to_cpc_network = flows_to_cpc[flows_to_cpc['Flow'].isin(model_flows + ['CONSTRUCTION', 'DECOMMISSION', 'PROCESS_FUEL', 'TRANSPORT_FUEL'])]

# Unit conversion: factors per unit of main output, as in the generic file
unit_conversion_network = adapt_to_network(unit_conversion[~unit_conversion['Type'].isin(['Flow', 'Other'])], lca_names)
unit_conversion_network = pd.concat([
    unit_conversion_network,
    unit_conversion[unit_conversion['Name'].isin(components)],
    unit_conversion[(unit_conversion['Type'] == 'Flow') & unit_conversion['Name'].isin(model_flows)],
    unit_conversion[unit_conversion['Type'] == 'Other'],
], ignore_index=True)

# Lifetime: ESM column from the network (technologies only, not components)
lifetime_network = adapt_to_network(lifetime, lca_names, keep=components)
lifetime_network['ESM'] = lifetime_network['Name'].map(tech_data['Lifetime'])
lifetime_network = lifetime_network[['Name', 'ESM', 'LCA']]

# Efficiency: combustion technologies only
efficiency = technologies[
    technologies['Category'].isin(['conversion', 'end use', 'demand'])
    & (technologies['Main input'].isin(combustion_fuels) | technologies.get('Combustion', False).fillna(False))
    & technologies['Name'].isin(mapping_network.loc[mapping_network['Type'] == 'Operation', 'Name'])
][['Name', 'Main input']].rename(columns={'Main input': 'Flow'})
efficiency['Flow'] = efficiency['Flow'].apply(lambda flow: str([flow]))

# The energy efficiency and lifetime of technologies can vary over years, so they're written in a year-specific folder
model.to_csv(f'{output_path}/{year}/model.csv', index=False)
lifetime_network.to_csv(f'{output_path}/{year}/lifetime.csv', index=False)

# The rest of the files are common to all years, so they're written in the main folder
efficiency.to_csv(f'{output_path}efficiency.csv', index=False)
mapping_network.to_csv(f'{output_path}mapping.csv', index=False)
unit_conversion_network.to_csv(f'{output_path}unit_conversion.csv', index=False)
compositions_network.to_csv(f'{output_path}technology_compositions.csv', index=False)
tech_specifics_network.to_csv(f'{output_path}technology_specifics.csv', index=False)
flows_to_cpc_network.to_csv(f'{output_path}mapping_esm_flows_to_CPC.csv', index=False)
technologies.to_csv(f'{output_path}technologies.csv', index=False)

print(f'{lca_names.notna().sum()} technologies mapped out of {len(lca_names)}, {len(model)} flows, '
      f'{len(efficiency)} technologies in the efficiency file, written to {output_path}')

82 technologies mapped out of 88, 163 flows, 28 technologies in the efficiency file, written to your_specific_input_files/


Conversion factors to apply when integrating the LCA indicators in PyPSA:

In [11]:
technologies.loc[technologies['Conversion'] != 1, ['Name', 'Reference flow', 'Main output', 'Conversion', 'Conversion source']].head()

,Name,Reference flow,Main output,Conversion,Conversion source
0,AC (Line),AC,AC,64.354245,mean length of the assets (64.35 km): PyPSA ne...
1,BEV charger,low voltage,EV battery,0.900000,efficiency (0.9 MWh EV battery per MWh low vol...
2,CCGT,gas,AC,0.587500,efficiency (0.5875 MWh AC per MWh gas): PyPSA ...
3,DAC,AC,co2 stored,1.818182,efficiency (1.818 t co2 stored per MWh AC): Py...
5,Fischer-Tropsch,H2,oil,0.730667,efficiency (0.7307 MWh oil per MWh H2): PyPSA ...
